# Пользователь и бд

```
# psql -U postgres
psql (17.6 (Debian 17.6-1.pgdg13+1))
Type "help" for help.

postgres=# CREATE USER quack WITH password 's3cr3t';
CREATE ROLE

postgres=# CREATE DATABASE quack_db OWNER quack;
CREATE DATABASE

postgres=# quit

# psql --user=quack quack_db
psql (17.6 (Debian 17.6-1.pgdg13+1))
Type "help" for help.

quack_db=> \dt
Did not find any relations.
quack_db=> \q
```

![](diagram.png)

# SQL

In [ ]:
CREATE TABLE users (
    id SERIAL PRIMARY KEY,
    username VARCHAR(50) NOT NULL UNIQUE,
    email VARCHAR(50) NOT NULL UNIQUE
);

-- (OneToOne), (ForeignKey)
CREATE TABLE user_profiles (
    id SERIAL PRIMARY KEY,
    user_id INTEGER NOT NULL UNIQUE, 
    bio TEXT,
    phone_number VARCHAR(20),
    CONSTRAINT fk_user_profile FOREIGN KEY (user_id) REFERENCES users(id) ON DELETE CASCADE
);

-- (ForeignKey)
CREATE TABLE courses (
    id SERIAL PRIMARY KEY,
    title VARCHAR(200) NOT NULL,
    description TEXT,
    author_id INTEGER NOT NULL,
    CONSTRAINT fk_course_author FOREIGN KEY (author_id) REFERENCES users(id) ON DELETE CASCADE
);

-- (ManyToMany), (ManyToOne)
CREATE TABLE course_students (
    user_id INTEGER NOT NULL,
    course_id INTEGER NOT NULL,
    PRIMARY KEY (user_id, course_id),
    CONSTRAINT fk_student FOREIGN KEY (user_id) REFERENCES users(id) ON DELETE CASCADE,
    CONSTRAINT fk_course FOREIGN KEY (course_id) REFERENCES courses(id) ON DELETE CASCADE
);

In [ ]:
SELECT * from users u

In [ ]:
SELECT app, name FROM django_migrations ORDER BY id DESC;

In [ ]:
SELECT * FROM user_profiles LIMIT 1;

In [ ]:
SELECT * from dbapp_admin;

# Код psycopg2

In [ ]:
import psycopg2
from psycopg2 import sql
import itertools
import sys
import json
import settings

def connect_db():
    try:
        conn = psycopg2.connect(**settings.DATABASES[default])
        return conn
    except Exception as e:
        print(f"Ошибка подключения к базе данных: {e}")
        sys.exit(1)

def insert_collection(conn, name, description):
    with conn.cursor() as cur:
        insert_query = """
            INSERT INTO collections (name, description)
            VALUES (%s, %s)
            ON CONFLICT (name) DO NOTHING
            RETURNING collection_id;
        """
        cur.execute(insert_query, (name, description))
        result = cur.fetchone()
        if result:
            collection_id = result[0]
            print(f"Создана новая коллекция с ID: {collection_id}")
        else:
            # Получить существующую коллекцию
            cur.execute("SELECT collection_id FROM collections WHERE name = %s;", (name,))
            collection_id = cur.fetchone()[0]
            print(f"Коллекция уже существует с ID: {collection_id}")
        return collection_id

def insert_attribute_rarity(conn, attributes):
    with conn.cursor() as cur:
        for attr_name, values in attributes.items():
            rarity = round(100.0 / len(values), 2)  # Равная редкость
            for val in values:
                insert_query = """
                    INSERT INTO attribute_rarity (attribute_name, attribute_value, rarity_percent)
                    VALUES (%s, %s, %s)
                    ON CONFLICT (attribute_name, attribute_value) DO NOTHING;
                """
                cur.execute(insert_query, (attr_name, val, rarity))
                print(f"Добавлен атрибут: {attr_name} - {val} с редкостью {rarity}%")
    conn.commit()

def get_admin_id(conn):
    with conn.cursor() as cur:
        cur.execute("SELECT id FROM users WHERE username = 'admin';")
        result = cur.fetchone()
        if result:
            admin_id = result[0]
            return admin_id
        else:
            raise Exception("Администратор не найден. Пожалуйста, создайте администратора перед запуском скрипта.")

def generate_combinations(attributes):
    # Сортируем атрибуты для консистентности
    sorted_attrs = sorted(attributes.items())
    attr_names = [attr[0] for attr in sorted_attrs]
    attr_values = [attr[1] for attr in sorted_attrs]
    # Генерируем картезианское произведение
    combinations = list(itertools.product(*attr_values))
    print(f"Сгенерировано {len(combinations)} комбинаций атрибутов.")
    return attr_names, combinations

def insert_nfts(conn, collection_id, admin_id, attr_names, combinations, image_url_base):
    with conn.cursor() as cur:
        img_num = 1
        for combo in combinations:
            image_url = f"{image_url_base}{img_num}.png"
            # Вставка NFT
            insert_nft_query = """
                INSERT INTO nfts (collection_id, owner_id, image_url, mint_date)
                VALUES (%s, %s, %s, CURRENT_TIMESTAMP)
                RETURNING nft_id;
            """
            cur.execute(insert_nft_query, (collection_id, admin_id, image_url))
            nft_id = cur.fetchone()[0]
            print(f"Вставлена NFT с ID: {nft_id}, Image URL: {image_url}")

            # Вставка атрибутов NFT
            nft_attrs = zip(attr_names, combo)
            insert_nft_attr_query = """
                INSERT INTO nft_attributes (nft_id, attribute_name, attribute_value)
                VALUES (%s, %s, %s)
                ON CONFLICT (nft_id, attribute_name) DO NOTHING;
            """
            for attr_name, attr_value in nft_attrs:
                cur.execute(insert_nft_attr_query, (nft_id, attr_name, attr_value))
                print(f" - Добавлен атрибут: {attr_name} = {attr_value}")

            img_num += 1
    conn.commit()
    print("Все NFT успешно вставлены.")

def main():
    if len(sys.argv) != 2:
        print("Использование: python insert_nft_collection.py <config.json>")
        sys.exit(1)
    
    config_file = sys.argv[1]
    try:
        with open(config_file, 'r', encoding='utf-8') as f:
            config = json.load(f)
    except Exception as e:
        print(f"Ошибка чтения конфигурационного файла: {e}")
        sys.exit(1)
    
    collection_name = config.get('name')
    collection_description = config.get('description')
    attributes = config.get('attributes')  # Ожидается словарь {attr_name: [values]}
    image_url_base = config.get('image_url_base')  # Например, 'https://example.com/images/nft'

    if not all([collection_name, collection_description, attributes, image_url_base]):
        print("Конфигурационный файл должен содержать 'name', 'description', 'attributes' и 'image_url_base'.")
        sys.exit(1)
    
    conn = connect_db()
    try:
        conn.autocommit = False  # Начало транзакции
        collection_id = insert_collection(conn, collection_name, collection_description)
        insert_attribute_rarity(conn, attributes)
        admin_id = get_admin_id(conn)
        attr_names, combinations = generate_combinations(attributes)
        insert_nfts(conn, collection_id, admin_id, attr_names, combinations, image_url_base)
        conn.commit()
        print("Инициализация коллекции завершена успешно.")
    except Exception as e:
        conn.rollback()
        print(f"Произошла ошибка: {e}")
    finally:
        conn.close()

if __name__ == "__main__":
    main()


Ошибка чтения конфигурационного файла: [Errno 2] No such file or directory: '--f=/home/thinkercat/.local/share/jupyter/runtime/kernel-v2-207831FBzHjKUYeC23.json'


SystemExit: 1

/home/thinkercat/Documents/DB-NFT/.venv/lib64/python3.13/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# Django ORM

python manage.py makemigrations

python manage.py migrate

python manage.py sqlmigrate dbapp 0001